In [6]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor

# Load cleaned dataset
df = pd.read_csv("../../data/processed/student_lifestyle_cleaned.csv")

# Encode stress level
stress_map = {
    "Low": 0,
    "Moderate": 1,
    "High": 2
}
df["Stress_Level"] = df["Stress_Level"].map(stress_map)

# Define features and target
feature_cols = [
    "Study_Hours_Per_Day",
    "Sleep_Hours_Per_Day",
    "Social_Hours_Per_Day",
    "Physical_Activity_Hours_Per_Day",
    "Extracurricular_Hours_Per_Day",
    "Stress_Level"
]

X = df[feature_cols]
y = df["GPA"]

# Shared split for everyone
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Task 3 - XGBoost Regressor

**Objective:** Build an advanced boosting model to predict GPA.

## Step 1 - Train XGBoost Regressor

In [7]:
# Train XGBoost Regressor
xgb_model = XGBRegressor(random_state=42, n_estimators=100, learning_rate=0.1)
xgb_model.fit(X_train, y_train)
print("XGBoost model trained successfully.")

XGBoost model trained successfully.


## Step 2 - 5-Fold Cross Validation (on Training Set)

In [8]:
# 5-Fold Cross Validation on training set
cv_scores = cross_val_score(xgb_model, X_train, y_train, cv=5, scoring="r2")

print("5-Fold CV R² scores:", np.round(cv_scores, 4))
print(f"Mean CV R²: {cv_scores.mean():.4f}")

5-Fold CV R² scores: [0.4275 0.4337 0.4324 0.4457 0.4903]
Mean CV R²: 0.4459


## Step 3 - Predict on Test Set & Evaluate

In [9]:
# Predict on test set
y_pred = xgb_model.predict(X_test)

# Evaluate
test_r2 = r2_score(y_test, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Test R²:  {test_r2:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")

Test R²:  0.4452
Test RMSE: 0.2276


## Step 4 - Save Model

In [10]:
# Save model for Task 5 (Feature Importance)
models_dir = "../../models"
os.makedirs(models_dir, exist_ok=True)

joblib.dump(xgb_model, os.path.join(models_dir, "xgboost_model.pkl"))
print(f"Model saved to {models_dir}/xgboost_model.pkl")

Model saved to ../../models/xgboost_model.pkl


## Summary

| Metric | Value |
|---|---|
| CV R² (mean) | 0.4459 |
| Test R² | 0.4452 |
| Test RMSE | 0.2276 |

**Explanation:**
The XGBoost model explains about 45% of the variation in GPA based on the 5 lifestyle features. The close match between CV R² (0.4459) and Test R² (0.4452) shows the model is stable and not overfitting. The RMSE of 0.2276 means predictions are off by roughly 0.23 GPA points on average. This suggests that daily habits like study hours, sleep, and social activity do influence GPA, but other unmeasured factors (e.g., study quality, course difficulty, motivation) account for the remaining 55%. The model was saved for use in Task 5 (Feature Importance).